In [31]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Conv2D,
    Conv2DTranspose,
    Dense,
    Reshape,
    Embedding,
    BatchNormalization,
)

tf.keras.backend.clear_session()

# =============================================================================
# 1. Conditional Generator Architecture
# =============================================================================
class ConditionalGenerator(Model):
    """cGAN generator: (z, label) -> 128x128x3."""
    def __init__(
        self,
        n_classes,
        latent_dim,
        embed_dim=128,
        base_channels=512,
        img_channels=3,
        **kwargs,
    ):
        super().__init__(**kwargs)
        
        # Names mapped exactly to your h5py deep scan output
        self.label_embedding = Embedding(n_classes, embed_dim, name="label_embedding")
        self.fc = Dense(4 * 4 * base_channels, name="fc")
        self.reshape = Reshape((4, 4, base_channels), name="reshape")

        self.g_tconv_8 = Conv2DTranspose(512, 4, strides=2, padding="same", use_bias=False, name="g_tconv_8")
        self.g_bn_8 = BatchNormalization(name="g_bn_8")

        self.g_tconv_16 = Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False, name="g_tconv_16")
        self.g_bn_16 = BatchNormalization(name="g_bn_16")

        self.g_tconv_32 = Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False, name="g_tconv_32")
        self.g_bn_32 = BatchNormalization(name="g_bn_32")

        self.g_tconv_64 = Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False, name="g_tconv_64")
        self.g_bn_64 = BatchNormalization(name="g_bn_64")
        
        self.g_tconv_128 = Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False, name="g_tconv_128")
        self.g_bn_128 = BatchNormalization(name="g_bn_128")

        self.g_out = Conv2D(img_channels, 7, padding="same", activation="tanh", name="g_out")

    def call(self, inputs, training=False):
        z, label = inputs
        le = self.label_embedding(label)
        le = tf.squeeze(le, axis=1)
        x = tf.concat([z, le], axis=-1)
        x = self.fc(x)
        x = self.reshape(x)

        x = tf.nn.relu(self.g_bn_8(self.g_tconv_8(x), training=training))
        x = tf.nn.relu(self.g_bn_16(self.g_tconv_16(x), training=training))
        x = tf.nn.relu(self.g_bn_32(self.g_tconv_32(x), training=training))
        x = tf.nn.relu(self.g_bn_64(self.g_tconv_64(x), training=training))
        x = tf.nn.relu(self.g_bn_128(self.g_tconv_128(x), training=training))

        return self.g_out(x)

In [32]:
# =============================================================================
# 2. The Stopwatch Function
# =============================================================================
def measure_cgan_inference_time(generator, latent_dim=128, n_classes=8, batch_size=32, num_runs=50):
    print(f"Batch size: {batch_size}")
    
    # Warm-up Phase
    print("Warming up the GPU (compiling graph & initializing memory)...")
    z_warmup = tf.random.normal((batch_size, latent_dim))
    labels_warmup = tf.random.uniform((batch_size, 1), minval=0, maxval=n_classes, dtype=tf.int32)
    _ = generator([z_warmup, labels_warmup], training=False).numpy()
    
    # Benchmark Phase
    print(f"Running {num_runs} consecutive batches to find the average speed.")
    
    start_time = time.time()
    
    for _ in range(num_runs):
        z = tf.random.normal((batch_size, latent_dim))
        labels = tf.random.uniform((batch_size, 1), minval=0, maxval=n_classes, dtype=tf.int32)
        
        # .numpy() forces the GPU to finish before Python moves to the next line
        _ = generator([z, labels], training=False).numpy()
        
    
    end_time = time.time()
    
    # Math calculations
    total_time = end_time - start_time
    time_per_batch = total_time / num_runs
    time_per_image = time_per_batch / batch_size
    images_per_second = 1.0 / time_per_image
    
    print("\n=== Benchmark Results ===")
    print(f"Total Time for {num_runs * batch_size} images: {total_time:.4f} seconds")
    print(f"Average time per batch ({batch_size} images): {time_per_batch:.4f} seconds")
    print(f"Average time per single image: {time_per_image:.5f} seconds")
    print(f"Generation Speed: {images_per_second:.2f} images / second")
    
    return images_per_second

# =============================================================================
# 3. Execution Setup & Benchmark
# =============================================================================
LATENT_DIM = 128
N_CLASSES = 8  
WEIGHTS_PATH = "/kaggle/input/models/ahmadhassan100100/gen-100/keras/default/1/generator_epoch_100.weights.h5"

# Initializing the empty Generator shell
generator = ConditionalGenerator(n_classes=N_CLASSES, latent_dim=LATENT_DIM)

# Dummy pass to map variables safely
print("Performing dummy pass to build layers correctly...")
dummy_z = tf.zeros((1, LATENT_DIM))
dummy_label = tf.zeros((1, 1), dtype=tf.int32)
_ = generator([dummy_z, dummy_label], training=False)

# Loadimg the trained weights
generator.load_weights(WEIGHTS_PATH)
print(f"Successfully loaded weights from {WEIGHTS_PATH}\n")

# Run the benchmark
speed_metric = measure_cgan_inference_time(
    generator=generator, 
    latent_dim=LATENT_DIM, 
    n_classes=N_CLASSES, 
    batch_size=16, 
    num_runs=5      
)

Performing dummy pass to build layers correctly...
Successfully loaded weights from /kaggle/input/models/ahmadhassan100100/gen-100/keras/default/1/generator_epoch_100.weights.h5

Batch size: 16
Warming up the GPU (compiling graph & initializing memory)...
Running 5 consecutive batches to find the average speed.

=== Benchmark Results ===
Total Time for 80 images: 0.1024 seconds
Average time per batch (16 images): 0.0205 seconds
Average time per single image: 0.00128 seconds
Generation Speed: 781.38 images / second
